In [ ]:
from torch.nn import *
from math import log2
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def center(a):
    return a.mul(2.0).add_(-1.0)


def dat(k, c, s=1, n=1, l=1, rx=1.0, rk=1.0, rb=1.0):
    l = 2 ** int(log2(c * s * (k + 1) * l))
    y = torch.randint(c * s, (n, l))
    return (
        torch.randn(n, c * s, k)
        .mul_(rx * rb * l ** (1 / k))[torch.arange(n)[:, None], y]
        .view(n * l, 1, k)
        .baddbmm(torch.randn(n * l, 1, k).mul_(rx), torch.randn(n, c * s, k, k)[torch.arange(n)[:, None], y].view(n * l, k, k))
        .view(n, l, k),
        unpack(y // s if s - 1 & s else y >> int(log2(s)), log2(c).__ceil__()),
    )


class Attention(Sequential):
    def forward(s, q, k=None, v=None):
        if k is None:
            k = q
        return (s[0](q) @ s[1](k).transpose(-2, -1)).softmax(-1) @ s[2](k if v is None else v)


class Norm(Module):
    @staticmethod
    def forward(x):
        return x * x.norm(dim=-1, keepdim=True)


class Scale(Module):
    def __init__(s, w=..., b=None):
        super().__init__()
        s.w = Parameter(torch.ones(())) if w is ... else w
        s.b = Parameter(torch.zeros(())) if b is ... else b

    def forward(s, x):
        if s.w is not None:
            x = x * s.w
        if s.b is not None:
            x += s.b
        return x


class LinearConst(Linear):
    def __init__(s, x, *_):
        super().__init__(x.size(-1), *_)
        s.x = x

    def forward(s, _):
        return super().forward(s.x)


class Add(Sequential):
    def forward(s, x):
        return s[0](x[0]) + s[1](x[1]) if isinstance(x, tuple) else s[0](x) + s[1](x)


class TrainEval(Module):
    def __init__(s, k, q, v, h=1, l=1, w=1):
        super().__init__()
        w = int(w * h * k)
        l = int(l * 2 ** v)
        h = int(h * k)
        s.i = center(unpack(torch.arange(l), log2(l).__ceil__()))
        s.a = Attention(Identity(), Sequential(Linear(k, h), Hardswish(1), Linear(h, s.i.size(1))), Sequential(Linear(k, h), Hardswish(1), *(Linear(h, w),) if w < h else ()))
        s.b = Attention(
            Sequential(Linear(q, h), Hardswish(1), Linear(h, h)),
            Sequential(Add(Linear(w, h, 0), LinearConst(s.i, h)), Hardswish(1)),
            Sequential(Add(Linear(w, h, 0), LinearConst(s.i, h)), Hardswish(1)),
        )
        s.c = Sequential(Add(Linear(h, h, 0), Linear(q, h)), Hardswish(1), Linear(h, v))

    def forward(s, k, q):
        return s.c((s.b(q, s.a(s.i, k)), q))

w/l|.5 | 1 | 2 | 4 | 8 |16
---|---|---|---|---|---|---
.5h1|14|15|18
 1 |15|23|50|56|73
 2 |21|31|70|57
 4 |50|62|80|91
 8 |74|75|87|97
16 ||84

In [ ]:
n = 6
m = TrainEval(2 * n, n, n, 16, 0.5)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.001, 100000, pct_start=0.01, div_factor=9, final_div_factor=999)
print(m, sum(i.numel() for i in m.parameters()))
for i in range(O.total_steps // 1000):
    for i in range(1000):
        x, y = dat(n, 1 << n, n=32)
        x = m(torch.cat((x, y), 2), x)
        i = functional.binary_cross_entropy_with_logits(x, y)
        i.backward()
        o.step()
        O.step()
        o.zero_grad()
    x = (x.signbit() != y).all(-1)
    print(i, x.sum() / x.numel(), o.param_groups[0]["lr"])